# Heart Disease Prediction
Exploratory data analysis and comparison of four classification models (Logistic Regression, KNN, Naive Bayes, Random Forest).

**Note:** place `heart_dataset.csv` in the same folder as this notebook, or change the path in the first code cell.

## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import plot_tree

%matplotlib inline

## 2. Load Dataset

In [ ]:
df = pd.read_csv("heart_dataset.csv")   # change path if needed

print("First five rows:")
display(df.head())

In [ ]:
print("Last five rows:")
display(df.tail())

print("\nDataset shape:", df.shape)

In [ ]:
print("Dataset information:")
df.info()

In [ ]:
print("Dataset description:")
display(df.describe())

In [ ]:
print("Data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

## Objective 1: Heart Disease Distribution

In [ ]:
print(df["target"].value_counts())

df["target"].value_counts().sort_index().plot(kind="bar")
plt.title("Heart Disease Distribution")
plt.xlabel("Target (0 = No Disease, 1 = Disease)")
plt.ylabel("Number of Patients")
plt.tight_layout()
plt.show()

## Objective 2: Age and Sex

In [ ]:
sns.histplot(df["age"], bins=10, kde=True)
plt.title("Age Distribution of Patients")
plt.xlabel("Age")
plt.ylabel("Number of Patients")
plt.tight_layout()
plt.show()

In [ ]:
sns.boxplot(x="target", y="age", data=df)
plt.title("Age vs Heart Disease")
plt.xlabel("Target (0 = No Disease, 1 = Disease)")
plt.ylabel("Age")
plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(df["sex"], df["target"]).plot(kind="bar")
plt.title("Sex vs Heart Disease")
plt.xlabel("Sex (0 = Female, 1 = Male)")
plt.ylabel("Number of Patients")
plt.legend(["No Disease", "Heart Disease"])
plt.tight_layout()
plt.show()

## Objective 3: Chest Pain and Exercise Angina

In [ ]:
pd.crosstab(df["chest pain type"], df["target"]).plot(kind="bar")
plt.title("Chest Pain Type vs Heart Disease")
plt.xlabel("Chest Pain Type")
plt.ylabel("Number of Patients")
plt.legend(["No Disease", "Heart Disease"])
plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(df["exercise angina"], df["target"]).plot(kind="bar")
plt.title("Exercise Angina vs Heart Disease")
plt.xlabel("Exercise Angina (0 = No, 1 = Yes)")
plt.ylabel("Number of Patients")
plt.legend(["No Disease", "Heart Disease"])
plt.tight_layout()
plt.show()

## Objective 4: Medical Measurements

In [ ]:
cols = ["resting bp s", "cholesterol", "max heart rate", "oldpeak"]

for col in cols:
    sns.boxplot(x="target", y=col, data=df)
    plt.title(f"{col} vs Heart Disease")
    plt.xlabel("Target (0 = No Disease, 1 = Disease)")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

## Objective 5: Correlation Analysis

In [ ]:
corr = df.corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Heart Dataset Correlation Heatmap")
plt.tight_layout()
plt.show()

## Prepare Data for Machine Learning

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

print("Features:")
display(X.head())
print("Target:")
display(y.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train shape:", X_train_s.shape, "| Test shape:", X_test_s.shape)

In [ ]:
def evaluate(name, model, Xtr, Xte):
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    print(name.upper())
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, pred))
    print("Classification Report:")
    print(classification_report(y_test, pred))
    return pred

## Model 1: Logistic Regression

In [ ]:
model1 = LogisticRegression(max_iter=1000)
y_pred1 = evaluate("Logistic Regression", model1, X_train_s, X_test_s)

## Model 2: K-Nearest Neighbors

In [ ]:
model2 = KNeighborsClassifier(n_neighbors=5, metric="euclidean")
y_pred2 = evaluate("K-Nearest Neighbors", model2, X_train_s, X_test_s)

## Model 3: Naive Bayes

In [ ]:
model3 = GaussianNB()
y_pred3 = evaluate("Naive Bayes", model3, X_train_s, X_test_s)

## Model 4: Random Forest

In [ ]:
model4 = RandomForestClassifier(n_estimators=100, random_state=42)
y_pred4 = evaluate("Random Forest", model4, X_train_s, X_test_s)

### Random Forest: Decision Tree Visualization

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(
    model4.estimators_[0],
    feature_names=X.columns,
    class_names=["No Disease", "Heart Disease"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8
)
plt.title("Random Forest - Decision Tree Visualization")
plt.show()

### Random Forest: Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model4.feature_importances_
}).sort_values(by="Importance", ascending=False)

display(feature_importance)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, x="Importance", y="Feature")
plt.title("Random Forest - Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

## Compare All Four Models

In [ ]:
results_df = pd.DataFrame({
    "Model": ["Logistic Regression", "KNN", "Naive Bayes", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred1),
        accuracy_score(y_test, y_pred2),
        accuracy_score(y_test, y_pred3),
        accuracy_score(y_test, y_pred4),
    ],
})

display(results_df)

best_model = results_df.loc[results_df["Accuracy"].idxmax()]
print("Best Model:", best_model["Model"])
print("Best Accuracy:", round(best_model["Accuracy"] * 100, 2), "%")

In [ ]:
results_df.plot(x="Model", y="Accuracy", kind="bar", legend=False)
plt.title("Model Accuracy Comparison")
plt.xlabel("Machine Learning Model")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()